# Proyecto 1
by Sergio Andres Gerena Gomez

# 1. Introducción

El presente proyecto cubre el proceso completo de extracción, transformación y análisis de información bibliográfica de una revista científica Q1. Se documenta el proceso de scraping junto con los distintos retos que presentó el sitio web, la construcción de una base de datos consultable mediante SQLite y los principales hallazgos obtenidos a partir de los datos recolectados.


La revista seleccionada fue el **Journal of Hematology & Oncology**, una publicación centrada en investigaciones de alto impacto sobre hematología y cáncer. Es un referente mundial en la revisión y publicación de investigaciones de vanguardia, con la participación de líderes en esos campos, y su enfoque principal son los resultados de investigaciones de laboratorio que buscan el aval de la comunidad científica a través de sus revisiones. Además de lo anterior, la revista es de acceso abierto y libre para todo público, lo que facilitó la obtención legal de la información extraída y posibilitó su análisis. El carácter experimental de sus publicaciones hace probable la aparición de investigaciones con componentes estadísticos, dado el uso frecuente de ensayos clínicos.

El enlace a la página de la revista es: https://link.springer.com/journal/13045


# 2. Fuente de datos y scraping

El ejercicio se enfocó en los artículos publicados en 2025. Al realizar una búsqueda en la página, se encontró que todas las publicaciones de ese año estaban agrupadas en el volumen 18, por lo que el enlace específico desde el que se extrajo la información fue: https://link.springer.com/journal/13045/volumes-and-issues/18-1?page=1

## Librerías
A continuación se presentan las librerías utilizadas:

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# selenium: automatización del navegador para la extracción de información de la página web
import time  # pausas entre páginas para evitar sobrecarga del servidor
import uuid  # generación de un identificador único por publicación
from datetime import datetime  # manejo y formato de fechas
import pandas as pd  # manipulación y análisis de datos en DataFrames
import sqlite3  # interfaz con bases de datos SQLite desde Python

A continuación, en este bloque de código se definen las funciones auxiliares utilizadas para el procesamiento y clasificación de la información extraída.

In [ ]:

# Diccionario con palabras clave por categoría temática, utilizado para clasificar cada artículo
# según el contenido de su resumen (abstract).
keywords = {
    "Machine Learning": [
        " machine learning ", " ml ", " deep learning ",
        " neural network ", " cnn ", " rnn ",
        " random forest ", " svm ", " xgboost ",
        " clustering ", " feature selection ",
        " predictive model ", " prediction model ",
        " training dataset ", " validation dataset "
    ],

    "IA Generativa": [
        " generative ai ", " llm ", " gpt ",
        " transformer ", " diffusion ",
        " text generation ", " image generation ",
        " large language model "
    ],

    "Estadística": [
        " statistical ", " statistics ",
        " hypothesis ", " p-value ",
        " confidence interval ",
        " anova ", " chi-square ",
        " logistic regression ",
        " regression analysis ",
        " cox regression ",
        " survival analysis ",
        " kaplan-meier ",
        " hazard ratio ",
        " odds ratio ",
        " bayesian ",
        " probability ",
        " distribution "
    ]
}
# Lista vacía donde se irán acumulando los registros de cada artículo como diccionarios
data = []

def normalize(text):
    """Convierte el texto a minúsculas para facilitar la comparación de palabras clave.

    Args:
        text (str): Texto original a normalizar.

    Returns:
        str: Texto convertido a minúsculas.
    """
    text = text.lower()
    return text

def classify_abstract(abstract):
    """Clasifica un artículo en una categoría temática según las palabras clave presentes en su abstract.

    Cuenta las coincidencias de palabras clave de cada categoría en el texto del abstract.
    La categoría con mayor número de coincidencias es asignada al artículo. Si no hay
    ninguna coincidencia, se asigna la etiqueta 'Otros'. Si el abstract está vacío o
    ausente, retorna 'no se pudo clasificar'.

    Args:
        abstract (str): Texto del resumen del artículo.

    Returns:
        str: Categoría temática asignada: 'Machine Learning', 'IA Generativa',
             'Estadística', 'Otros' o 'no se pudo clasificar'.
    """
    if not abstract or (isinstance(abstract, str) and abstract.strip() == ""):
        return "no se pudo clasificar"
    
    text = normalize(abstract)
    scores = {k: 0 for k in keywords}
    for category, words in keywords.items():
        for w in words:
            if w in text:
                scores[category] += 1
    best_category = max(scores, key=scores.get)
    if scores[best_category] == 0:
        return "Otros"
    
    return best_category

def parse_number(value):
    """Convierte un valor numérico a entero, interpretando el sufijo 'k' como miles.

    Algunos contadores de citas y descargas en la página web presentan valores
    abreviados con la letra 'k' (por ejemplo, '2.3k' equivale a 2300).
    Esta función estandariza esos valores a su representación entera.

    Args:
        value (str | int | float): Valor numérico, posiblemente con sufijo 'k'.

    Returns:
        int: Valor numérico entero correspondiente.
    """
    value = str(value).strip().lower()
    if value.endswith("k"):
        return int(float(value[:-1]) * 1000)
    return int(value)

# Scraping

Para el proceso de scraping, se realizó primero una extracción de todos los enlaces a artículos presentes en la página actual. Dado que el listado estaba paginado (los 114 artículos no se mostraban en una sola página), se implementó un bucle que, al finalizar el recorrido de los enlaces de la página actual, buscara el botón "Siguiente" y lo presionara. Si dicho botón no era encontrado o no era interactuable, se interpretaba como el final del volumen y se detenía el proceso.

A continuación se detalla el código.

In [ ]:
driver = webdriver.Chrome()  # instancia el driver para controlar el navegador Chrome
wait = WebDriverWait(driver, 10)  # tiempo máximo de espera antes de lanzar un timeout

url = "https://link.springer.com/journal/13045/volumes-and-issues/18-1?page=1"
driver.get(url)

# Bloque para aceptar el aviso de cookies si aparece al cargar la página
try:
    cookie_button = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-cc-action='accept']"))
    )
    cookie_button.click()
    print("Cookies aceptadas")
except:
    print("No apareció banner de cookies")

# Bucle principal de scraping: recorre todas las páginas del volumen
while True:
    wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "section[data-test='article-listing']"))
    )  # espera a que el listado de artículos esté disponible en el DOM

    articles = driver.find_elements(
        By.CSS_SELECTOR,
        "section[data-test='article-listing'] h2 a"
    )  # obtiene todos los elementos de enlace de artículos visibles en pantalla

    links = [a.get_attribute("href") for a in articles]
    # extrae las URLs de cada artículo para acceder luego a su información completa

    print(f"Links encontrados: {len(links)}")

    # Recorre cada enlace y extrae los campos de interés del artículo
    for link in links:
        driver.get(link)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        print("Visitando:", link)

        journal_element = driver.find_element(By.XPATH, """//*[@id="main"]/section/div/div/div[2]/a[1]/span""")
        title_element = driver.find_element(By.XPATH, """//*[@id="main"]/section/div/div/div[1]/h1""")
        title = title_element.text
        # Omite las entradas que corresponden a correcciones o retractaciones de artículos anteriores
        if title.startswith("Retraction Note:") or title.startswith("Correction:"):
            driver.back()
            wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "section[data-test='article-listing']"))
            )
            continue
        date_element = driver.find_element(By.XPATH, """//*[@id="main"]/section/div/div/div[1]/ul[1]/li[3]/time""")
        items = driver.find_elements(By.CSS_SELECTOR, "li.c-bibliographic-information__list-item")
        doi = None
        # Busca el DOI dentro de la sección de información bibliográfica
        for item in items:
            if item.find_elements(By.CSS_SELECTOR, 'abbr[title="Digital Object Identifier"]'):
                doi = item.find_element(By.CSS_SELECTOR, "span.c-bibliographic-information__value").text
                break
        abstract_element = driver.find_elements(By.CSS_SELECTOR, "#Abs1-content")
        authors_element = driver.find_elements(By.CSS_SELECTOR, "li.c-article-author-list__item")
        authors_list = []
        # Extrae únicamente los nombres de autores reales, filtrando notas o texto no relacionado
        for a in authors_element:
            try:
                author_tag = a.find_element(By.CSS_SELECTOR, "a[data-test='author-name']")
                authors_list.append(author_tag.text.strip())
            except:
                pass
        citations_element = driver.find_elements(By.CSS_SELECTOR, """li[data-test="citation-count"] p""")
        downloads_element = driver.find_elements(By.CSS_SELECTOR, """li[data-test="access-count"] p""")
        refs_elements = driver.find_elements(By.CSS_SELECTOR, "p.c-article-references__text")
        refs_list = [ref.text.strip() for ref in refs_elements]
        parsed_date = datetime.strptime(date_element.text, "%d %B %Y")
        
        paper_id = uuid.uuid4()
        journal_name = journal_element.text
        publication_date = parsed_date.strftime("%Y-%m-%d")
        year = parsed_date.year
        url = link
        abstract = abstract_element[0].text if abstract_element else ""
        authors = ",".join(authors_list)
        n_authors = len(authors_list)
        citations = parse_number(citations_element[0].text.split()[0] if citations_element else "0")
        downloads = parse_number(downloads_element[0].text.split()[0] if downloads_element else "0")
        references = "<REF>".join(refs_list)
        n_references = len(refs_list)
        topic_label = classify_abstract(abstract)
        record = {
            "paper_id": str(paper_id),
            "journal_name": journal_name,
            "title": title,
            "publication_date": publication_date,
            "year": year,
            "doi": doi,
            "url": url,
            "abstract": abstract,
            "authors": authors,
            "n_authors": n_authors,
            "citations": citations,
            "downloads": downloads,
            "references": references,
            "n_references": n_references,
            "topic_label": topic_label
        }

        data.append(record)
        driver.back()
        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "section[data-test='article-listing']"))
        )

    try:
        next_button = driver.find_element(By.CSS_SELECTOR, "a[rel='next']")

        # Si el botón está deshabilitado, se ha llegado a la última página
        if "disabled" in next_button.get_attribute("class"):
            print("No hay más páginas.")
            break

        next_button.click()

        # Pausa para que la nueva página cargue completamente antes de continuar
        time.sleep(2)

    except:
        print("No se encontró botón Next. Fin.")
        break

driver.quit()

Cookies aceptadas
Links encontrados: 50
Visitando: https://link.springer.com/article/10.1186/s13045-025-01770-7
Visitando: https://link.springer.com/article/10.1186/s13045-025-01763-6
Visitando: https://link.springer.com/article/10.1186/s13045-025-01764-5
Visitando: https://link.springer.com/article/10.1186/s13045-025-01768-1
Visitando: https://link.springer.com/article/10.1186/s13045-025-01749-4
Visitando: https://link.springer.com/article/10.1186/s13045-025-01750-x
Visitando: https://link.springer.com/article/10.1186/s13045-025-01757-4
Visitando: https://link.springer.com/article/10.1186/s13045-025-01734-x
Visitando: https://link.springer.com/article/10.1186/s13045-025-01767-2
Visitando: https://link.springer.com/article/10.1186/s13045-025-01759-2
Visitando: https://link.springer.com/article/10.1186/s13045-025-01760-9
Visitando: https://link.springer.com/article/10.1186/s13045-025-01756-5
Visitando: https://link.springer.com/article/10.1186/s13045-025-01751-w
Visitando: https://link.

# 3. Construcción de la base SQLite

Una vez recopilada la lista de diccionarios con todas las publicaciones válidas, se procede a transformarla en un DataFrame de pandas y a persistirla en una base de datos SQLite. El siguiente código muestra dicha operación.

In [8]:
papers = pd.DataFrame(data)
conn = sqlite3.connect("grupo_2/gomez_sergio/parcial_1/revista_q1_2025.sqlite")
papers.to_sql("papers", conn, if_exists="replace", index=False)
conn.close()
papers

,paper_id,journal_name,title,publication_date,year,doi,url,abstract,authors,n_authors,citations,downloads,references,n_references,topic_label
0,3cd10984-8d7c-4c70-97d3-bfd0ef961b4e,Journal of Hematology & Oncology,Naxitamab plus stepped-up dosing of granulocyt...,2025-11-26,2025,https://doi.org/10.1186/s13045-025-01770-7,https://link.springer.com/article/10.1186/s130...,"Background\nWith high-risk neuroblastoma, post...","Brian H. Kushner,Shakeel Modak,Audrey Mauguen,...",6,2,1295,"Berthold F, Faldum A, Ernst A, Boos J, Dilloo ...",47,Otros
1,7c4b45d8-d1ba-4c48-abf7-269683aa8484,Journal of Hematology & Oncology,Inflammasomes and pyroptosis in cancer: mechan...,2025-11-25,2025,https://doi.org/10.1186/s13045-025-01763-6,https://link.springer.com/article/10.1186/s130...,Inflammasomes are being increasingly recognize...,"Wonhyoung Seo,Bokeum Jung,Taylor Roh,Hyo Jung ...",7,5,1700,"Liu Z, Xu S, Chen L, Gong J, Wang M. The role ...",307,Otros
2,25857d51-e45c-4655-8f7e-0f0e741e11b3,Journal of Hematology & Oncology,Updated data of CLL1 CAR-T cell therapy in adu...,2025-12-18,2025,https://doi.org/10.1186/s13045-025-01764-5,https://link.springer.com/article/10.1186/s130...,CLL1-targeted chimeric antigen receptor T (CAR...,"Xiaomei Zhang,Wenyi Lu,Wenjun Zhang,Xiaoyuan H...",23,0,3408,"Tashiro H, Sauer T, Shum T, Parikh K, Mamonkin...",6,Otros
3,540f37d6-d355-4f06-9427-363cfd31df25,Journal of Hematology & Oncology,PACIFIC-5: a phase III clinical trial of conso...,2025-12-07,2025,https://doi.org/10.1186/s13045-025-01768-1,https://link.springer.com/article/10.1186/s130...,Background\nConsolidation durvalumab following...,"Yi-Long Wu,Lin Wu,Nan Bi,Timucin Cil,Hong Ge,Z...",20,3,6460,"Bray F, Laversanne M, Sung H, Ferlay J, Siegel...",48,Estadística
4,6aba7d34-7b73-4562-8e80-40d7cfb2550b,Journal of Hematology & Oncology,Aberrant KIF26B expression promotes bladder ca...,2025-12-22,2025,https://doi.org/10.1186/s13045-025-01749-4,https://link.springer.com/article/10.1186/s130...,Abnormal expression of kinesins has been obser...,"Jia-Ming Wang,Hai-Yun Xie,Feng-Hao Zhang,Xuan ...",7,1,3237,"Chen X, Li A, Sun BF, Yang Y, Han YN, Yuan X, ...",7,Otros
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,37676770-66f9-4c03-85e2-0c962e1f1093,Journal of Hematology & Oncology,Global status and attributable risk factors of...,2025-01-10,2025,https://doi.org/10.1186/s13045-025-01660-y,https://link.springer.com/article/10.1186/s130...,"Background\nFemale-specific cancers, particula...","Tianye Li,Haoxiang Zhang,Mengyi Lian,Qionghua ...",9,154,19000,"Bray F, Laversanne M, Sung H, Ferlay J, Siegel...",32,Otros
100,e2614903-12a9-49ce-ba93-f8a271fd27f6,Journal of Hematology & Oncology,Impact of circulating lymphoma cells at diagno...,2025-01-05,2025,https://doi.org/10.1186/s13045-024-01658-y,https://link.springer.com/article/10.1186/s130...,"Diffuse large B-cell lymphoma (DLBCL), the mos...","Sayan Mullick Chowdhury,Subodh Bhatta,Timothy ...",15,2,2828,"Li S, Young KH, Medeiros LJ. Diffuse large B-c...",8,Otros
101,8bebb1da-2ddf-4add-b5a8-e3c6594367ae,Journal of Hematology & Oncology,A phase II trial of anlotinib plus EGFR-TKIs i...,2025-01-05,2025,https://doi.org/10.1186/s13045-024-01656-0,https://link.springer.com/article/10.1186/s130...,Background\nThe study is to evaluate the effic...,"Hua-Jun Chen,Hai-Yan Tu,Yanping Hu,Yun Fan,Guo...",18,7,4507,"Huang L, Jiang S, Shi Y. Tyrosine kinase inhib...",43,Otros
102,2f45bfc7-dc6a-497c-a39a-c7cc19860c85,Journal of Hematology & Oncology,D-ribose-5-phosphate inactivates YAP and funct...,2025-01-04,2025,https://doi.org/10.1186/s13045-024-01655-1,https://link.springer.com/article/10.1186/s130...,Background\nTargeting glucose uptake by glucos...,"Cheng-E Tu,Yong-Feng Liu,Hong-Wei Liu,Chun-Mei...",10,4,5029,"Ohshima K, Morii E. Metabolic reprogramming of...",51,Otros


# 4. Clasificación temática

Cada artículo fue etiquetado automáticamente utilizando la función `classify_abstract`, que aplica un criterio de conteo de palabras clave sobre el texto del abstract. Se definieron tres categorías principales (**Machine Learning**, **IA Generativa** y **Estadística**) mediante diccionarios de términos representativos de cada área. La categoría con mayor número de coincidencias es asignada al artículo; si no hay ninguna, se etiqueta como **Otros**. En caso de que el abstract no esté disponible, se registra como **no se pudo clasificar**.

# 5. Consultas SQL y resultados

En la siguiente sección se establece una conexión a la base de datos SQLite generada anteriormente para responder las preguntas de análisis planteadas.

In [ ]:
conn = sqlite3.connect('grupo_2/gomez_sergio/parcial_1/revista_q1_2025.sqlite')

## 1. ¿Cuál es el número promedio de autores por paper?

In [10]:
promedio_autores = pd.read_sql_query("SELECT ROUND(AVG(n_authors), 0) AS promedio FROM papers", conn)
promedio_autores

,promedio
0,13.0


El promedio de autores por artículo es de **13**, lo que refleja el carácter colaborativo de las investigaciones en esta revista.

## 2. ¿Cuántos artículos están relacionados con Machine Learning?

In [11]:
n_ml = pd.read_sql_query("SELECT COUNT(*) AS n_art_ml FROM papers WHERE topic_label = 'Machine Learning'", conn)
n_ml

,n_art_ml
0,0


En la revista no se encontró ningún artículo clasificado bajo la categoría de Machine Learning.

## 3. ¿Cuántos artículos están relacionados con IA Generativa?

In [12]:
n_ia_gen = pd.read_sql_query("SELECT COUNT(*) AS n_art_iagen FROM papers WHERE topic_label = 'IA Generativa'", conn)
n_ia_gen

,n_art_iagen
0,0


Tampoco se encontraron artículos clasificados bajo la categoría de IA Generativa.

## 4. ¿Cuántos artículos están relacionados con otros temas estadísticos?

In [13]:
n_est = pd.read_sql_query("SELECT COUNT(*) AS n_est FROM papers WHERE topic_label = 'Estadística'", conn)
n_est

,n_est
0,6


Se encontraron **6 publicaciones** relacionadas con estadística en este volumen de la revista.

## 5. ¿Cuál es el número total de descargas de los artículos publicados en 2025?

In [14]:
t_desc = pd.read_sql_query("SELECT SUM(downloads) AS total_descargas FROM papers", conn)
t_desc

,total_descargas
0,858455


Los artículos de la revista publicados en 2025 acumularon un total de **858.455 descargas y visualizaciones**, lo que evidencia el alto nivel de difusión de la publicación.

## 6. ¿Cuál es el número promedio de referencias por artículo?

In [15]:
avg_ref = pd.read_sql_query("SELECT ROUND(AVG(n_references), 0) AS average_references FROM papers", conn)
avg_ref

,average_references
0,130.0


El promedio de referencias por artículo en los publicados durante 2025 es de **130**, un número elevado que refleja el rigor bibliográfico de la revista.

## 7. ¿Cuál es la referencia que más se repite entre todos los artículos?

In [16]:
mod_ref = pd.read_sql_query("""
                            WITH RECURSIVE split(valor, resto) AS (
                                SELECT
                                  '',
                                  [references] || '<REF>'
                                FROM papers

                                UNION ALL

                                 SELECT
                                   SUBSTR(resto, 0, INSTR(resto, '<REF>')),
                                   SUBSTR(resto, INSTR(resto, '<REF>') + 5)
                                 FROM split
                                 WHERE resto != ''
                            )
                            SELECT
                              valor     AS referencia,
                              COUNT(*)  AS frecuencia
                            FROM split
                            WHERE valor != ''
                            GROUP BY valor
                            ORDER BY frecuencia DESC
                            LIMIT 1""", conn)
pd.set_option('display.max_colwidth', None)
mod_ref

,referencia,frecuencia
0,"Cappell KM, Kochenderfer JN. Long-term outcomes following CAR T cell therapy: what we know so far. Nat Rev Clin Oncol. 2023;20(6):359–71.",4


La referencia más citada entre las publicaciones de 2025 fue el artículo de Cappell y Kochenderfer sobre los resultados a largo plazo de la terapia con células CAR T, con un total de **4 apariciones** en distintos artículos del volumen.

## 8. ¿Cuál es el promedio de citas por artículo?

In [17]:
avg_cit = pd.read_sql_query("SELECT ROUND(AVG(citations), 0) AS average_citations FROM papers", conn)
avg_cit

,average_citations
0,18.0


El promedio de citaciones por artículo es de **18**, considerando que muchos de ellos fueron publicados recientemente y el índice de citas aún está en crecimiento.

## 9. ¿Cuál es el paper con más citas?

In [18]:
top_citated_paper = pd.read_sql_query("""SELECT title, 
                                      citations 
                                      FROM papers 
                                      ORDER BY citations DESC 
                                      LIMIT 1""", conn)
top_citated_paper

,title,citations
0,Helicobacter pylori and gastric cancer: mechanisms and new perspectives,194


El artículo más citado de la revista en 2025 fue **"Helicobacter pylori and gastric cancer: mechanisms and new perspectives"**, con **194 citaciones**.

## 10. ¿Cuál es el paper relacionado con Machine Learning, IA Generativa o Estadística con más citas?

In [19]:
top_est_citated_paper = pd.read_sql_query("""SELECT title, 
                                          citations,
                                          topic_label
                                          FROM papers 
                                          WHERE topic_label IN ('Machine Learning', 'IA Generativa', 'Estadística')
                                          ORDER BY citations DESC 
                                          LIMIT 1""", conn)
top_est_citated_paper

,title,citations,topic_label
0,Treatment-related adverse events of antibody drug-conjugates in clinical trials,22,Estadística


Dentro de las categorías temáticas de interés, el artículo con más citas fue **"Treatment-related adverse events of antibody drug-conjugates in clinical trials"**, clasificado bajo Estadística, con **22 citas**.

## 11. ¿Cuál es el paper relacionado con Machine Learning, IA Generativa o Estadística con más descargas?

In [20]:
top_est_downloaded_paper = pd.read_sql_query("""SELECT title, 
                                          downloads,
                                          topic_label
                                          FROM papers 
                                          WHERE topic_label IN ('Machine Learning', 'IA Generativa', 'Estadística')
                                          ORDER BY downloads DESC 
                                          LIMIT 1""", conn)
top_est_downloaded_paper

,title,downloads,topic_label
0,Treatment-related adverse events of antibody drug-conjugates in clinical trials,7356,Estadística


El mismo artículo sobre eventos adversos en ensayos clínicos también lidera en descargas dentro de las categorías de interés, con **7.356 descargas**.

In [21]:
conn.close()

# 6. Hallazgos

Uno de los resultados más llamativos fue la proporción de publicaciones que, al momento de la extracción, no correspondían a artículos originales. El volumen 18 contaba inicialmente con 114 entradas, pero el DataFrame final conservó únicamente 104, lo que representa una depuración del **8,8%**. Este descarte se debió a que dichas entradas eran correcciones o retractaciones de artículos anteriores, las cuales carecen de la estructura informativa habitual (abstract, autoría estándar, métricas propias) y, por lo tanto, no cumplían con los criterios del análisis.

Otro hallazgo relevante es que, pese a la vocación clínica de la revista, apenas **6 de los 104 artículos** válidos (un 5.8%) presentaron mención explícita a términos estadísticos en sus abstracts. La búsqueda incluyó conceptos directamente asociados a ensayos clínicos, como odds ratio, regresión logística o análisis de supervivencia. El restante 94.2% de los artículos exhibió un enfoque marcadamente experimental, centrado en mecanismos moleculares, terapias oncológicas y hallazgos de laboratorio, sin una terminología estadística formal en sus resúmenes.

# 7. Limitaciones

Se presentaron varias dificultades a lo largo del proceso. La primera, ya mencionada en la sección de hallazgos, fue la existencia de entradas que no correspondían a artículos originales sino a correcciones o retractaciones. Al carecer de la estructura común y no aportar la información necesaria para el análisis, se decidió excluirlas del conjunto de datos final.

Una segunda limitación fue la asignación incorrecta de clases CSS a ciertos elementos del sitio web. En algunos artículos, textos que eran notas editoriales aparecían etiquetados con la misma clase que los nombres de autores, lo que interrumpía la extracción de esa información. Dado que la distribución y el orden de los elementos en el DOM no eran homogéneos entre artículos (por ejemplo, las secciones del abstract podían variar en profundidad de anidamiento), se optó por un enfoque basado en selectores de clase CSS en lugar de rutas XPath absolutas. Para los casos con etiquetado incorrecto, el bloque de extracción de autores se envolvió en un bloque `try/except` que simplemente omite los elementos que no cumplen con la estructura esperada.

Por último, se encontró un artículo en particular cuyo contenido no estaba dividido en secciones estándar: carecía de abstract, metodología y desarrollo diferenciados, asemejándose más a una monografía. Para dicho artículo fue imposible extraer un resumen, por lo que el campo `abstract` se dejó vacío y su `topic_label` quedó registrada explícitamente como `no se pudo clasificar`. El registro se conservó en la base de datos dado que el resto de sus campos sí eran accesibles.

# 8. Conclusión

Este proyecto demostró que es posible construir un pipeline completo de extracción, almacenamiento y análisis de información bibliográfica a partir de una revista científica de acceso abierto, utilizando herramientas de código abierto como Selenium, pandas y SQLite.

La revista **Journal of Hematology & Oncology** se confirmó como una publicación de orientación predominantemente experimental y clínica, con un perfil estadístico presente pero no dominante: apenas el 5.8% de los artículos del volumen 18 hicieron referencia explícita a métodos estadísticos en sus abstracts, y ninguno fue clasificado bajo Machine Learning o IA Generativa. Esto refleja que, aunque los ensayos clínicos son frecuentes en la revista, el énfasis narrativo de los resúmenes se concentra en resultados biomédicos más que en la metodología cuantitativa empleada.

Desde la perspectiva técnica, el scraping con Selenium resultó adecuado para un sitio con contenido dinámico y paginación, aunque requirió manejo cuidadoso de excepciones ante la falta de estandarización estructural entre artículos. La estrategia de clasificación por conteo de palabras clave sobre el abstract es funcional y transparente, aunque su sensibilidad depende directamente de la riqueza del vocabulario utilizado en los resúmenes; una mejora natural sería incorporar el texto completo del artículo o aplicar modelos de lenguaje para una categorización más robusta.